# CITYKIN WO4 — Societies-tab PCA-cluster replacement: Step 1 validation

**Scope of this notebook: Step 1 only** — engine generalization checks, before Karl's sign-off gate
ahead of Step 2 (the API path). Two things to establish:

1. **Terrain-variable provenance** — confirm `distance_core.py`'s `terrain` lens (`relief_range_m`,
   `landform_position`) reads the point-window `dplace.society_terrain` columns, not a basin-aggregate
   substitute (WO2a's area confound would transfer if it did).
2. **Reproduce WO8d's EA034 numbers** through the new parameterized `scan()` path — must match the
   hand-run Part C table in `wo8d_findings.md` exactly (same substrate, same seed/n_draws).
3. **EA042 calibration run** — confirm a subsistence value with a known strong environmental
   association actually lights up on `water`/`thermal`, on **both** cohesion and displacement. This is
   the check that the instrument detects real signal at all; a scan with no known-positive is
   uninterpretable.

**Engine:** `scripts/cdop/distance_core.py` — WO4's additions (`displacement`, `random_draw_stats`,
`top_families`, `scan`) sit alongside WO8d's original functions, unchanged, so WO8d's own notebook
keeps reproducing independently of this one. Unit tests: `tests/cdop/test_distance_core.py`, 20 green
(10 original WO8d + 10 new).

**Design settled, not re-litigated here:** family-restricted resampling is deliberately not part of
this scan (Karl + Opus, 2026-07-30 — "environmental similarity net family" is a TRACE-phase analytical
move, not this screen's job); the composition note is a plain top-3-family count/share, no baseline.
Full reasoning: `docs/cdop/citykin/wo4_whc-grouping.md`, `docs/cdop/citykin/note_to_opus_societies-dataviz.md`.

WO: `docs/cdop/citykin/wo4_whc-grouping.md`. Prior: `wo8d_env-culture-highgods.md`, `wo8a_findings.md` Part B.

In [1]:
# Cell 1
%matplotlib inline
from pathlib import Path

import numpy as np
import pandas as pd

import scripts.shared.db_utils as db_utils
from scripts.cdop.distance_core import (
    LENSES, backdrop_z, cohesion, random_draw_cohesions, percentile_rank,
    displacement, random_draw_stats, displacement_percentile_rank, top_families, scan,
)

ROOT = Path(db_utils.__file__).parent.parent.parent
OUT  = ROOT / 'output' / 'cdop'
SUBSTRATE_PATH = OUT / 'wo8c_substrate.parquet'

sub = pd.read_parquet(SUBSTRATE_PATH)
print(f"substrate: {SUBSTRATE_PATH.name} | rows={len(sub)} | columns={len(sub.columns)}")
print(f"has ea042_subsistence: {'ea042_subsistence' in sub.columns} | has ea034_religion: {'ea034_religion' in sub.columns}")

substrate: wo8c_substrate.parquet | rows=1133 | columns=35
has ea042_subsistence: True | has ea034_religion: True


In [2]:
# Cell 2 -- Step 1 check 1: terrain-variable provenance. The substrate carries BOTH a point-window
# relief column (`relief_range_m`, from `dplace.society_terrain` / `persist_dplace_terrain.py`'s
# +-2km/1km grid sample) and a separately-named basin-aggregate one (`basin_relief_range_m`). If the
# `terrain` lens in LENSES is reading `relief_range_m`, it's on the point-window column by construction
# (name match) -- this cell exists to make that fact visible in output, not just readable in source,
# and to check the two columns actually differ (if they were identical, something upstream would be
# suspect -- point-window and basin-aggregate relief should NOT agree closely in general).
print("LENSES['terrain'] columns:", LENSES['terrain'])
print()
has_basin_col = 'basin_relief_range_m' in sub.columns
print(f"substrate also carries 'basin_relief_range_m' (basin-aggregate, WO2a's confounded variant): {has_basin_col}")

if has_basin_col:
    both = sub[['relief_range_m', 'basin_relief_range_m']].dropna()
    corr = both['relief_range_m'].corr(both['basin_relief_range_m'])
    print(f"n with both columns present: {len(both)}")
    print(f"correlation, point-window relief_range_m vs. basin-aggregate basin_relief_range_m: {corr:.3f}")
    print("(a real correlation is expected -- both measure relief in the same place -- but they should")
    print(" not be near-identical; a lens using relief_range_m is reading the point-window column,")
    print(" confirmed by name match against LENSES and by the two columns' distinct, non-identical values.)")
    print()
    print(both.head(8).to_string())

print()
print("landform_position null rate:", sub['landform_position'].isna().mean())
print("relief_range_m null rate:", sub['relief_range_m'].isna().mean())
print("(landform_position IS computed for D-PLACE societies -- dplace.society_terrain carries it,")
print(" persist_dplace_terrain.py's landform = (grid_elev_mean - grid_elev_min) / relief_range_m --")
print(" the terrain lens's second facet is real and point-window, not missing or basin-derived.)")
print()
print("CAVEAT (not a defect): dplace.society_terrain uses WO8c's original +-2km/1km grid, never")
print("updated to CITYKIN WO1a's later-corrected +-10km/5km box used for the WH Cities corpus.")
print("This scan never compares a society to a WH City directly, so it's immaterial here -- but the")
print("two `terrain` lenses (society-scan vs. WH-Cities-retrieval) are not on the same sampling window.")

LENSES['terrain'] columns: ['relief_range_m', 'landform_position']

substrate also carries 'basin_relief_range_m' (basin-aggregate, WO2a's confounded variant): True
n with both columns present: 1133
correlation, point-window relief_range_m vs. basin-aggregate basin_relief_range_m: 0.689
(a real correlation is expected -- both measure relief in the same place -- but they should
 not be near-identical; a lens using relief_range_m is reading the point-window column,
 confirmed by name match against LENSES and by the two columns' distinct, non-identical values.)

   relief_range_m  basin_relief_range_m
0           435.0                  2454
1           512.0                  1861
2           246.0                  2506
3           690.0                  1588
4            24.0                   509
5           268.0                  1156
6           208.0                   537
7           551.0                  1462

landform_position null rate: 0.00794351279788173
relief_range_m null rate

In [3]:
# Cell 3 -- Step 1 check 2: reproduce WO8d's EA034 Part C table through the parameterized scan().
# Reference values below are copied verbatim from wo8d_findings.md Part C (Cell 7 of the WO8d
# notebook) -- NOT re-derived here, so this cell's job is purely comparison.
REFERENCE = {
    # lens: (n_backdrop, obs_cohesion, random_draw_mean, pct_tighter_than_random)
    'water':   (1133, 0.540, 0.731, 95.25),
    'thermal': (1133, 1.223, 1.201, 44.75),
    'overall': (1133, 1.441, 1.525, 70.80),
    'terrain': (1124, 1.207, 1.184, 44.10),
}
FOCUS_CODE = 'Active, but not supporting morality'   # EA034-3, WO8d's exact focus class

out = scan(sub, trait_col='ea034_religion', value=FOCUS_CODE, n_draws=2000, seed=0)
print(f"n_focus_input (rows matching '{FOCUS_CODE}' before per-lens completeness): {out['n_focus_input']}")
print()

rows = []
for lens, ref in REFERENCE.items():
    res = out['lenses'][lens]
    rows.append({
        'lens': lens,
        'n_backdrop': res['n_backdrop'], 'ref_n_backdrop': ref[0],
        'obs_cohesion': round(res['obs_cohesion'], 3), 'ref_obs_cohesion': ref[1],
        'random_mean': round(res['random_cohesion_mean'], 3), 'ref_random_mean': ref[2],
        'pct_tighter': round(res['pct_tighter_than_random'], 2), 'ref_pct_tighter': ref[3],
    })
cmp = pd.DataFrame(rows).set_index('lens')
print(cmp.to_string())
print()
print("Reading: every 'ref_*' column should match its neighbor exactly (obs_cohesion is a deterministic")
print("function of the real 40 focus societies -- no RNG involved -- so it must match to the printed")
print("precision regardless of anything else). random_mean/pct_tighter depend on the n_draws=2000 seed=0")
print("resampling and should also match if the substrate row order is unchanged since WO8d ran.")

n_focus_input (rows matching 'Active, but not supporting morality' before per-lens completeness): 40

         n_backdrop  ref_n_backdrop  obs_cohesion  ref_obs_cohesion  random_mean  ref_random_mean  pct_tighter  ref_pct_tighter
lens                                                                                                                           
water          1133            1133         0.540             0.540        0.731            0.731        95.25            95.25
thermal        1133            1133         1.223             1.223        1.201            1.201        44.75            44.75
overall        1133            1133         1.441             1.441        1.525            1.525        70.80            70.80
terrain        1124            1124         1.207             1.207        1.184            1.184        44.10            44.10

Reading: every 'ref_*' column should match its neighbor exactly (obs_cohesion is a deterministic
function of the real 40 focus so

In [5]:
# Cell 4 -- Step 1 check 2b: same reproduction, on the REAL backdrop rather than synthetic data --
# confirms random_draw_stats's cohesion column is identical to random_draw_cohesions's, the property
# already unit-tested on synthetic data (tests/cdop/test_distance_core.py). If this cell disagrees with
# the unit test, something about the real data (dtype, NaN handling, row order) breaks the equivalence
# the synthetic test couldn't catch.
#
# NB: unlike scan(), backdrop_z() is called directly here, so `ari_log` isn't derived automatically --
# scan() only derives it on its own internal copy of `sub`, which never propagates back to this
# notebook's `sub` variable. Deriving it here explicitly (matching WO8d's own Cell 3 pattern) rather
# than mutating `sub` inside scan() and relying on that side effect.
if 'ari_log' not in sub.columns:
    sub['ari_log'] = np.log1p(sub['ari_ix_sav'])

ok, Xz = backdrop_z(sub, 'water')
fmask = (ok['ea034_religion'] == FOCUS_CODE).to_numpy()
k = int(fmask.sum())

coh_only = random_draw_cohesions(Xz, k=k, n_draws=2000, seed=0)
coh_joint, disp_joint = random_draw_stats(Xz, k=k, n_draws=2000, seed=0)

identical = np.array_equal(coh_only, coh_joint)
print(f"k (focus n on 'water' lens): {k}")
print(f"random_draw_cohesions and random_draw_stats produce identical cohesion arrays: {identical}")
if not identical:
    print(f"  max abs diff: {np.abs(coh_only - coh_joint).max()}")

k (focus n on 'water' lens): 40
random_draw_cohesions and random_draw_stats produce identical cohesion arrays: True


In [6]:
# Cell 5 -- Step 1 check 3: EA042 calibration. 'Pastoralism' is WO8a's own named example of a
# distinctive climate-envelope region ("warm-dry pastoralists", Part B) -- a subsistence strategy with
# a specific, named, expected position on exactly the water+thermal axes. If this doesn't light up on
# BOTH cohesion and displacement, the scan is not measuring what it claims to (WO4 Step 1 accept bar).
CALIBRATION_VALUE = 'Pastoralism'

out_ea042 = scan(sub, trait_col='ea042_subsistence', value=CALIBRATION_VALUE, n_draws=2000, seed=0)
print(f"EA042 calibration: '{CALIBRATION_VALUE}', n_focus_input={out_ea042['n_focus_input']}")
print()

rows = []
for lens, res in out_ea042['lenses'].items():
    rows.append({
        'lens': lens, 'n_focus': res['n_focus'],
        'obs_cohesion': round(res['obs_cohesion'], 3),
        'pct_tighter_than_random': round(res['pct_tighter_than_random'], 2),
        'obs_displacement': round(res['obs_displacement'], 3),
        'displacement_pct_rank': round(res['displacement_pct_rank'], 2),
    })
print(pd.DataFrame(rows).set_index('lens').to_string())
print()
print("Expected (WO8a Part B, 'warm-dry pastoralists'): high pct_tighter_than_random AND high")
print("displacement_pct_rank on 'water' and/or 'thermal' -- both statistics, not cohesion alone.")
print("If neither lights up, STOP -- do not proceed to Step 2 -- and report back rather than")
print("re-picking a calibration value until something looks right.")
print()
print("Composition note (for comparison against the scan display's format, not part of the calibration bar):")
print(out_ea042['composition'])

EA042 calibration: 'Pastoralism', n_focus_input=76

         n_focus  obs_cohesion  pct_tighter_than_random  obs_displacement  displacement_pct_rank
lens                                                                                            
water         76         1.158                     0.00             1.454                 100.00
thermal       76         1.207                    48.95             0.333                  97.55
overall       76         1.814                     0.05             1.492                 100.00
terrain       76         0.905                    99.95             0.362                  99.30

Expected (WO8a Part B, 'warm-dry pastoralists'): high pct_tighter_than_random AND high
displacement_pct_rank on 'water' and/or 'thermal' -- both statistics, not cohesion alone.
If neither lights up, STOP -- do not proceed to Step 2 -- and report back rather than
re-picking a calibration value until something looks right.

Composition note (for comparison against 

In [7]:
# Cell 6 -- Step 1 check 3b (contrast case, not required by the WO, cheap extra confidence): 'Intensive
# agriculture' is WO8a's other named region ("agriculture holds the wet-mild quadrant") -- if the
# instrument is real, this should show a DIFFERENT displacement direction/lens profile than Pastoralism,
# not just 'also lights up somewhere'. Composition note included for both to eyeball plausibility.
CONTRAST_VALUE = 'Intensive agriculture'

out_contrast = scan(sub, trait_col='ea042_subsistence', value=CONTRAST_VALUE, n_draws=2000, seed=0)
print(f"EA042 contrast: '{CONTRAST_VALUE}', n_focus_input={out_contrast['n_focus_input']}")
print()
rows = []
for lens, res in out_contrast['lenses'].items():
    rows.append({
        'lens': lens, 'n_focus': res['n_focus'],
        'obs_cohesion': round(res['obs_cohesion'], 3),
        'pct_tighter_than_random': round(res['pct_tighter_than_random'], 2),
        'obs_displacement': round(res['obs_displacement'], 3),
        'displacement_pct_rank': round(res['displacement_pct_rank'], 2),
    })
print(pd.DataFrame(rows).set_index('lens').to_string())
print()
print("composition:", out_contrast['composition'])

EA042 contrast: 'Intensive agriculture', n_focus_input=265

         n_focus  obs_cohesion  pct_tighter_than_random  obs_displacement  displacement_pct_rank
lens                                                                                            
water        265         0.700                    78.10             0.230                  100.0
thermal      265         1.121                    94.90             0.083                   72.9
overall      265         1.442                    95.95             0.245                   99.9
terrain      261         1.259                    10.75             0.105                   83.7

composition: {'n_total': 265, 'n_unresolved': 13, 'top_families': [{'family_id': 'atla1278', 'n': 51, 'share': 0.19245283018867926}, {'family_id': 'afro1255', 'n': 49, 'share': 0.18490566037735848}, {'family_id': 'indo1319', 'n': 34, 'share': 0.12830188679245283}]}


## What to check before signing off Step 1

1. **Cell 2** — does `relief_range_m` read as genuinely distinct from `basin_relief_range_m` (correlated
   but not near-identical), and does `landform_position` have a low null rate? If the correlation is
   suspiciously high (~0.95+), flag it — that would suggest the point-window measure isn't adding much
   independent information over the basin-aggregate one, worth knowing even if it doesn't block Step 1.
2. **Cell 3** — do all four `ref_*` columns match their neighbors? `obs_cohesion`/`ref_obs_cohesion`
   should match exactly (no RNG involved). If `random_mean`/`pct_tighter` don't match while
   `obs_cohesion` does, the generalization is *structurally* correct but something about draw-order
   reproducibility needs another look (worth flagging, not silently accepting a near-miss).
3. **Cell 4** — confirms Cell 3's RNG-reproduction property on real (not synthetic) data.
4. **Cell 5** — does Pastoralism actually light up on `water` and/or `thermal`, on *both* cohesion and
   displacement? This is the accept bar for the whole calibration check — a clear miss here means
   stopping, not adjusting the calibration case until something works.
5. **Cell 6** — does the contrast case (Intensive agriculture) show a genuinely different profile than
   Pastoralism, not just "also positive somewhere"?

Report back with what you see — Step 2 (the API path) doesn't start until this is signed off.